# 03 Roboflow audit (M1b-3)
Runs on Colab (Drive mounted). **Audit only: no roles are decided here (that is M1b-4).** For the three Roboflow zips it measures, per set and per split (train / valid / test): counts and labels, source grouping, signs of augmentation, near-duplicates (inside each set and across sets), plate size, and saves contact sheets to Drive.

In simple words: Roboflow can "augment" a dataset, i.e. make several altered copies of each photo (rotated, noisy, blurred). If copies of one photo end up in both train and test, the test score is inflated. This notebook puts numbers on how much of that happened.

**Image outputs:** the cells that show images (the two contact-sheet cells and the top-random-pairs cell in step 6b) are tagged `remove_output` (images are CC BY and the repo is public). Text and table outputs stay. See the last cell before committing.

**What this notebook cannot verify** (it says so again where it matters): that a file-name "source" is really one original photo; that a flagged image is truly augmented (the scores are evidence, not proof); that a set is US or EU or that plates are readable (only your eyes on the contact sheets tell); duplicates that are mirrored (ORB is not flip-invariant); licences.

In [58]:
import csv, math, pickle, random, re, shutil, time, zipfile
from collections import Counter
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from google.colab import drive

In [85]:
# Parameters
SEED = 0
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
REPORTS = f'{DRIVE}/reports/roboflow_audit'  # CSV caches and contact sheets (private Drive folder, never the repo)
LOCAL = '/content/rf'                        # zips are copied and extracted here only
# set name -> zip file name in ARCHIVES. EDIT these to the real file names (the setup cell lists what is there).
ZIPS = {
    'samrat_v5': 'License Plates.v5-augmented-license-plates.yolov8.zip',  # plate + vehicle classes
    'florida_v6': 'License Plate Recognition.v6i.yolov8.zip',              # Objects In The Wild v6
    'usa_v1': 'License-plate-usa.v1i.yolov8.zip',                          # license-plate-usa v1
}
SPLITS = ('train', 'valid', 'test')
# thresholds (explained next to the step that uses them)
SP_THRESH = 0.005       # salt-and-pepper: share of isolated black/white pixels above this = flagged
BORDER_THRESH = 0.05    # black border: share of the outer 3% frame that is near-black above this = flagged
BLUR_THRESH = 100.0     # blur: variance of the Laplacian BELOW this = flagged (common convention, not a law)
DHASH_MAX = 6           # dHash Hamming distance <= this = near-identical (PLAN.md rule)
ORB_FEATURES = 500      # ORB keypoints per image
ORB_RATIO = 0.75        # Lowe ratio test
ORB_MIN_GOOD = 8        # need this many good matches before the geometry check is run
ORB_KEEP = 8            # pairs with at least this many geometry-consistent matches are written to the CSV
ORB_FLOOR = 12          # the decision threshold is never below this (see controls)
N_CTRL = 30             # images for the positive controls
N_NEG = 20000           # random cross-split, different-source pairs per set for the negative control
TRIM_FRAC = 0.005       # T_ORB rule: assume up to this share of the random pairs are real duplicates, ignore them, put T_ORB just above the highest remaining score (never below ORB_FLOOR)
T_ORB_OVERRIDE = {'samrat_v5': 42, 'florida_v6': 25, 'usa_v1': 40}     # e.g. {'usa_v1': 15} replaces the rule's T_ORB for that set (applied in 5e-3; fa_rate is recomputed at that T)
N_SHEET = 20            # images per set per split on a contact sheet
N_PAIR_TOP = 30         # top-scoring flagged pairs per set and split pair on the pair sheets
RUN_ORB = True         # False: no ORB work at all (steps 1-4, dHash, contact sheets). True: also descriptors + timing estimate (phase 2)
ORB_LONG = True        # with RUN_ORB True: True also runs the controls and the all-pairs search (phase 3), after you have read the estimate

## Setup

In [60]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

Already up to date.


In [62]:
# Not in the top imports cell on purpose: the package only exists after the install cell above.
from alpr import ccpd  # ImportError on the first run? Restart the kernel once.
print('OpenCV', cv2.__version__)  # preinstalled on Colab; not added to pyproject.toml
print('zips on Drive:', sorted(p.name for p in Path(ARCHIVES).glob('*.zip')))
for k, v in ZIPS.items():
    assert Path(ARCHIVES, v).exists(), f'{k}: {v} not found in {ARCHIVES}; edit ZIPS in the parameters cell'
Path(REPORTS).mkdir(parents=True, exist_ok=True)

OpenCV 5.0.0
zips on Drive: ['License Plate Recognition.v6i.yolov8.zip', 'License Plates.v5-augmented-license-plates.yolov8.zip', 'License-plate-usa.v1i.yolov8.zip']


In [63]:
_last = {}
def progress(label: str, i: int, n: int, t0: float, every: float = 30.0) -> None:
    """Print 'label: i/n, elapsed' at most every `every` seconds (and at the end), so no long loop is silent."""
    now = time.time()
    if now - _last.get(label, 0.0) >= every or i == n:
        _last[label] = now
        print(f'  {label}: {i}/{n}, {now - t0:.0f} s elapsed', flush=True)

## Step 0: copy, MD5, extract
Each zip is copied to the VM, its MD5 is printed (record it in PLAN.md) and compared with the Drive copy, then it is extracted under `/content/rf/<set>/`. The zips are only read, never changed. Python's `zipfile` refuses absolute paths and `..` in member names.

In [64]:
ROOT = {}
for name, fn in ZIPS.items():
    src, dst = Path(ARCHIVES, fn), Path(LOCAL, fn)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        print(f'copying {fn} ({src.stat().st_size / 1e6:.0f} MB) from Drive ...', flush=True)
        shutil.copyfile(src, dst)
    md5 = ccpd.file_md5(dst)
    assert md5 == ccpd.file_md5(src), f'{name}: VM copy differs from the Drive file'
    ROOT[name] = Path(LOCAL, name)
    if not ROOT[name].exists():
        with zipfile.ZipFile(dst) as z:
            z.extractall(ROOT[name])
    print(f'{name}: MD5 {md5}  splits found: {[s for s in SPLITS if (ROOT[name] / s / "images").is_dir()]}')

samrat_v5: MD5 13e58f5a41f8e2e79375e9944c66dd4d  splits found: ['train', 'valid', 'test']
florida_v6: MD5 5970cdf35e4dbe188f2c82de92536ecb  splits found: ['train', 'valid', 'test']
usa_v1: MD5 47ab70118bd47344bacba2c1efe4101a  splits found: ['train', 'valid', 'test']


## Step 1: what the README says, counts, label format, classes
The README files hold no images, so they are printed in full: they say which pre-processing and augmentation Roboflow applied. Then, per split: images, label files, images without a label file, and the label-line format (5 fields = a box, more = a polygon). Every label line is checked: integer class id in range, coordinates between 0 and 1. For a set with more than one class (samrat v5: plate + vehicle) boxes are counted per class and images left **without any plate box** are counted.

In [ ]:
import yaml  # preinstalled on Colab (PyYAML)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def readme(name):
    return '\n'.join(p.read_text(errors='replace') for p in sorted(ROOT[name].glob('README*.txt')))
CLASSES, PLATE_IDS = {}, {}
for name in ZIPS:
    print('=' * 20, name, '=' * 20)
    print(readme(name) or '(no README*.txt found)')
    y = yaml.safe_load((ROOT[name] / 'data.yaml').read_text())
    names = y['names']
    CLASSES[name] = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
    PLATE_IDS[name] = {i for i, n in CLASSES[name].items() if 'plate' in str(n).lower()}
    assert PLATE_IDS[name], f'{name}: no class name contains "plate": {CLASSES[name]}'
    print('classes:', CLASSES[name], '-> plate ids:', PLATE_IDS[name])

In [66]:
def images_of(name, split):
    return sorted(p for p in (ROOT[name] / split / 'images').iterdir() if p.suffix.lower() in IMG_EXT)

def read_boxes(label: Path, n_cls: int) -> tuple[list[tuple[int, float, float, float, float]], Counter]:
    """Boxes (cls, cx, cy, w, h) from one YOLO label file + a Counter of problems. A polygon line is turned into its bounding box."""
    boxes, bad, fmt = [], Counter(), Counter()
    for line in label.read_text().splitlines():
        f = line.split()
        if not f:
            continue
        try:
            cls = int(f[0]); v = [float(x) for x in f[1:]]
        except ValueError:
            bad['not_numeric'] += 1; continue
        if not (0 <= cls < n_cls):
            bad['class_out_of_range'] += 1; continue
        if len(v) == 4:
            cx, cy, w, h = v; fmt['box5'] += 1
        elif len(v) >= 6 and len(v) % 2 == 0:
            xs, ys = v[0::2], v[1::2]; fmt['polygon'] += 1
            cx, cy, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
        else:
            bad['bad_field_count'] += 1; continue
        if not all(0 <= t <= 1 for t in (cx, cy, w, h)) or w <= 0 or h <= 0:
            bad['coord_out_of_0_1'] += 1; continue
        boxes.append((cls, cx, cy, w, h))
    return boxes, bad + fmt

BOXES = {}   # (set, split, image name) -> list of boxes
rows = []
for name in ZIPS:
    for split in SPLITS:
        if not (ROOT[name] / split / 'images').is_dir():
            print(f'{name}/{split}: folder absent'); continue
        imgs = images_of(name, split)
        nolabel, tot, per_cls, noplate = 0, Counter(), Counter(), 0
        for p in imgs:
            lab = ROOT[name] / split / 'labels' / (p.stem + '.txt')
            if not lab.exists():
                nolabel += 1; BOXES[(name, split, p.name)] = []; noplate += 1; continue
            b, info = read_boxes(lab, len(CLASSES[name]))
            tot += info; BOXES[(name, split, p.name)] = b
            per_cls.update(CLASSES[name][c[0]] for c in b)
            noplate += not any(c[0] in PLATE_IDS[name] for c in b)
        n_lab = len(list((ROOT[name] / split / 'labels').glob('*.txt')))
        rows.append(dict(set=name, split=split, images=len(imgs), label_files=n_lab, img_no_label=nolabel,
                         img_no_plate=noplate, boxes=dict(per_cls), format=dict(tot)))
print(pd.DataFrame(rows).to_string(index=False))

       set split  images  label_files  img_no_label  img_no_plate                                    boxes                        format
 samrat_v5 train     979          979             0             0 {'license-plate': 1303, 'vehicle': 1867}                {'box5': 3170}
 samrat_v5 valid      70           70             0             0    {'license-plate': 84, 'vehicle': 134}                 {'box5': 218}
 samrat_v5  test      35           35             0             0     {'license-plate': 45, 'vehicle': 67}                 {'box5': 112}
florida_v6 train     972          972             0            58                  {'License-Plate': 1371}                {'box5': 1371}
florida_v6 valid      92           92             0             0                   {'License-Plate': 147}                 {'box5': 147}
florida_v6  test      46           46             0             4                    {'License-Plate': 61}                  {'box5': 61}
    usa_v1 train    1806         1806    

## Step 2: source grouping
Roboflow names files `<original name>.rf.<hash>.jpg`. The part before `.rf.` is the original upload, so augmented copies of one photo share it. Per split: distinct sources, copies per source (min / median / max), and how many sources appear in more than one split. **Limit:** if one photo was uploaded twice under two names, this step cannot see it (step 5 can). Files without `.rf.` are counted and treated as their own source.

In [67]:
def source_of(fname: str) -> str:
    return fname.split('.rf.')[0] if '.rf.' in fname else Path(fname).stem
SRC = {}  # set -> split -> Counter(source)
rows = []
for name in ZIPS:
    SRC[name] = {s: Counter(source_of(p.name) for p in images_of(name, s)) for s in SPLITS if (ROOT[name] / s / 'images').is_dir()}
    for s, c in SRC[name].items():
        others = set().union(*[set(o) for t, o in SRC[name].items() if t != s]) if len(SRC[name]) > 1 else set()
        v = sorted(c.values())
        rows.append(dict(set=name, split=s, images=sum(v), sources=len(c), copies_min=v[0], copies_median=float(np.median(v)), copies_max=v[-1],
                         sources_also_in_other_split=len(set(c) & others),
                         no_rf_in_name=sum(1 for p in images_of(name, s) if '.rf.' not in p.name)))
print(pd.DataFrame(rows).to_string(index=False))

       set split  images  sources  copies_min  copies_median  copies_max  sources_also_in_other_split  no_rf_in_name
 samrat_v5 train     979      245           3            4.0           4                            0              0
 samrat_v5 valid      70       70           1            1.0           1                            0              0
 samrat_v5  test      35       35           1            1.0           1                            0              0
florida_v6 train     972       97           3            6.0         459                           40              0
florida_v6 valid      92       41           1            1.0          45                           29              0
florida_v6  test      46       25           1            1.0          21                           23              0
    usa_v1 train    1806      578           3            3.0           9                            6              0
    usa_v1 valid      34       34           1            1.0    

## Step 3: signs of augmentation (three scores)
Each score is a number per image; the table gives the median and the share of images past the threshold. **Train is the positive reference** (the READMEs say it is augmented); valid and test are supposed to be clean.

| Score | What it measures, in simple words | Flag when | Why this threshold |
|---|---|---|---|
| **salt-and-pepper** | Share of pixels that are pure black or pure white AND very different from the median of their 3x3 neighbours. A lone black/white dot is noise; clean photos have almost none. | above `SP_THRESH` (0.5% of pixels) | checked on the control below (clean images vs the same images with added noise) |
| **black border** | Share of the outer 3% frame of the image that is near-black. Rotation or shifting leaves black wedges at the edges. | above `BORDER_THRESH` (5%) | checked on the control (images rotated 3, 10, 20 degrees; the control table shows how far past 5% each lands) |
| **blur** | Variance of the Laplacian (an edge-strength measure): sharp images have large values, blurry ones small. | below `BLUR_THRESH` (100) | a common rule of thumb, not a law; the control shows what added blur does |

**Limits:** a photo with a natural black edge or a night scene can fire the border score; an originally blurry photo fires the blur score, so blur **cannot prove** augmentation; crop, flip, brightness, exposure, hue and similar changes have no score here (see the capability table at the end).

The pass below also computes the dHash of the whole image and of the largest plate box (used in step 5). Results are appended to `scores.csv` on Drive every 100 images, so a disconnect loses nothing; a re-run only processes missing images.

In [68]:
def dhash(gray: Image.Image) -> int:  # same 64-bit difference hash as notebooks/01_eda.ipynb: left > right on a 9x8 thumbnail
    px = np.asarray(gray.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    return int(''.join('1' if b else '0' for b in (px[:, 1:] > px[:, :-1]).ravel()), 2)

def scores(im: Image.Image) -> dict:
    rgb = np.asarray(im.convert('RGB'))
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    med = cv2.medianBlur(g, 3)
    extreme = (g <= 10) | (g >= 245)
    sp = float(((np.abs(g.astype(np.int16) - med) > 64) & extreme).mean())
    h, w = g.shape
    k = max(1, int(0.03 * min(h, w)))
    dark = rgb.max(axis=2) <= 10
    frame = np.ones_like(dark); frame[k:h - k, k:w - k] = False
    return dict(sp=sp, border=float(dark[frame].mean()), blur=float(cv2.Laplacian(g, cv2.CV_64F).var()))

def plate_box(name, split, fname):
    b = [c for c in BOXES.get((name, split, fname), []) if c[0] in PLATE_IDS[name]]
    return max(b, key=lambda c: c[3] * c[4]) if b else None

SCORES_CSV = Path(REPORTS, 'scores.csv')
FIELDS = ['set', 'split', 'name', 'source', 'w', 'h', 'status', 'sp', 'border', 'blur', 'dhash', 'plate_dhash']
done = set()
if SCORES_CSV.exists():
    old = pd.read_csv(SCORES_CSV, dtype=str)
    done = set(zip(old['set'], old['split'], old['name']))
new = not SCORES_CSV.exists()
todo = [(n, s, p) for n in ZIPS for s in SPLITS if (ROOT[n] / s / 'images').is_dir() for p in images_of(n, s) if (n, s, p.name) not in done]
print(f'{len(done)} images already in scores.csv, {len(todo)} to do')
t_scores = time.time()
with open(SCORES_CSV, 'a', newline='') as f:
    wr = csv.DictWriter(f, FIELDS)
    if new:
        wr.writeheader()
    for i, (n, s, p) in enumerate(todo, 1):
        row = dict(set=n, split=s, name=p.name, source=source_of(p.name), status='ok')
        try:
            with Image.open(p) as im:
                im.load(); im = im.convert('RGB')
            row.update(w=im.width, h=im.height, **scores(im), dhash=format(dhash(im.convert('L')), '016x'))
            pb = plate_box(n, s, p.name)
            if pb:
                _, cx, cy, bw, bh = pb
                box = [max(0, int((cx - bw / 2) * im.width)), max(0, int((cy - bh / 2) * im.height)), min(im.width, int((cx + bw / 2) * im.width)), min(im.height, int((cy + bh / 2) * im.height))]
                if box[2] - box[0] >= 4 and box[3] - box[1] >= 4:
                    row['plate_dhash'] = format(dhash(im.crop(box).convert('L')), '016x')
        except (OSError, ValueError) as e:
            row['status'] = type(e).__name__  # never the message: it can hold the file name
        wr.writerow(row)
        if i % 100 == 0:
            f.flush()
        progress('scores', i, len(todo), t_scores)
df = pd.read_csv(SCORES_CSV, dtype={'dhash': str, 'plate_dhash': str})
print('status counts:', df['status'].value_counts().to_dict())
ok = df[df['status'] == 'ok'].copy()
ok['f_sp'] = ok['sp'] > SP_THRESH
ok['f_border'] = ok['border'] > BORDER_THRESH
ok['f_blur'] = ok['blur'] < BLUR_THRESH
ok['f_any'] = ok[['f_sp', 'f_border', 'f_blur']].any(axis=1)
g = ok.groupby(['set', 'split'])
print(pd.DataFrame({
    'images': g.size(), 'sp_median': g['sp'].median().round(4), 'sp_flagged': g['f_sp'].mean().round(3),
    'border_median': g['border'].median().round(4), 'border_flagged': g['f_border'].mean().round(3),
    'blur_median': g['blur'].median().round(1), 'blur_flagged': g['f_blur'].mean().round(3),
    'any_flag': g['f_any'].mean().round(3)}).to_string())

4068 images already in scores.csv, 0 to do
status counts: {'ok': 4068}
                  images  sp_median  sp_flagged  border_median  border_flagged  blur_median  blur_flagged  any_flag
set        split                                                                                                   
florida_v6 test       46     0.0257       0.891         0.3640           0.848      24175.4         0.000     1.000
           train     972     0.0152       0.832         0.2541           0.614      14078.0         0.001     0.939
           valid      92     0.0227       0.804         0.0814           0.533      17668.4         0.000     0.891
samrat_v5  test       35     0.0000       0.000         0.0022           0.143       1057.4         0.000     0.143
           train     979     0.0000       0.000         0.0050           0.356        872.1         0.000     0.356
           valid      70     0.0000       0.000         0.0027           0.200       1022.7         0.000     0.200
u

### Step 3b: controls for the three scores
Before trusting a threshold, test it on images where we **know** what was done. 30 random valid images (seeded) are altered one way at a time and scored again: added noise (3% of pixels set to black or white), rotated 3, 10 and 20 degrees with black fill (like Roboflow), and Gaussian blur (sigma 1.5). A good score moves clearly past its threshold for the altered copy and stays on the other side for the original.

In [69]:
def altered(im: Image.Image, rng: random.Random, rot: float = 0, crop: float = 1.0, noise: float = 0, blur: float = 0) -> Image.Image:
    if blur:
        im = im.filter(ImageFilter.GaussianBlur(blur))
    if rot:
        im = im.rotate(rot, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if crop < 1:
        w, h = im.size; cw, ch = int(w * crop), int(h * crop)
        x, y = rng.randint(0, w - cw), rng.randint(0, h - ch)
        im = im.crop((x, y, x + cw, y + ch)).resize((w, h), Image.BILINEAR)
    if noise:
        a = np.array(im); nrng = np.random.default_rng(rng.randint(0, 2**31))
        m = nrng.random(a.shape[:2]) < noise
        a[m] = nrng.choice([0, 255], size=(int(m.sum()), 1))
        im = Image.fromarray(a)
    return im

def ctrl_images(name):
    pool = [(s, p) for s in ('valid', 'test', 'train') if (ROOT[name] / s / 'images').is_dir() for p in images_of(name, s)]
    first = [x for x in pool if x[0] == pool[0][0]]  # valid if present
    return random.Random(SEED).sample(first, min(N_CTRL, len(first)))

rows = []
for name in ZIPS:
    print(f'score controls {name} ...', flush=True)
    imgs = [Image.open(p).convert('RGB') for _, p in ctrl_images(name)]
    rng = random.Random(SEED)
    variants = {'original': imgs, 'noise 3%': [altered(i, rng, noise=0.03) for i in imgs],
                'rot 3': [altered(i, rng, rot=3) for i in imgs], 'rot 10': [altered(i, rng, rot=10) for i in imgs],
                'rot 20': [altered(i, rng, rot=20) for i in imgs], 'blur 1.5': [altered(i, rng, blur=1.5) for i in imgs]}
    for v, ims in variants.items():
        sc = pd.DataFrame([scores(i) for i in ims])
        rows.append(dict(set=name, variant=v, n=len(ims), sp_median=sc.sp.median(), sp_flagged=(sc.sp > SP_THRESH).mean(),
                         border_median=sc.border.median(), border_flagged=(sc.border > BORDER_THRESH).mean(),
                         blur_median=sc.blur.median(), blur_flagged=(sc.blur < BLUR_THRESH).mean()))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print('Read: the row "original" should be mostly unflagged; the matching altered row should be mostly flagged for ITS score. If not, the threshold is wrong for that set: change it in the parameters cell.')

score controls samrat_v5 ...
score controls florida_v6 ...
score controls usa_v1 ...
       set  variant  n  sp_median  sp_flagged  border_median  border_flagged  blur_median  blur_flagged
 samrat_v5 original 30     0.0000      0.0000         0.0044          0.2333    1051.9392        0.0000
 samrat_v5 noise 3% 30     0.0224      1.0000         0.0198          0.2667   14348.9936        0.0000
 samrat_v5    rot 3 30     0.0000      0.0000         0.2268          1.0000     740.0466        0.0000
 samrat_v5   rot 10 30     0.0000      0.0000         0.4390          1.0000     793.4991        0.0000
 samrat_v5   rot 20 30     0.0000      0.0000         0.5056          1.0000     911.9398        0.0000
 samrat_v5 blur 1.5 30     0.0000      0.0000         0.0018          0.1667      21.2835        1.0000
florida_v6 original 30     0.0243      0.7667         0.0328          0.4333   17580.3510        0.0000
florida_v6 noise 3% 30     0.0443      1.0000         0.0469          0.5000   2848

## Step 4: plate box size
Plate box width, height and area as % of the image (largest plate box per image, boxes from the labels). Median plate width in pixels is `normalised width x image width` **in the exported image**, which Roboflow stretched to 416 or 640: the original photo's pixel width and aspect ratio are not recoverable, so read these as "size in the image a model would see".

In [70]:
rows = []
for (n, s, fname), b in BOXES.items():
    pb = plate_box(n, s, fname)
    r = ok[(ok['set'] == n) & (ok['split'] == s) & (ok['name'] == fname)]
    if pb and len(r):
        _, _, _, bw, bh = pb
        rows.append(dict(set=n, split=s, w_pct=100 * bw, h_pct=100 * bh, area_pct=100 * bw * bh, w_px=bw * float(r['w'].iloc[0]), img_w=int(r['w'].iloc[0]), img_h=int(r['h'].iloc[0])))
pl = pd.DataFrame(rows)
print(pl.groupby(['set', 'split']).agg(n=('w_pct', 'size'), w_pct=('w_pct', 'median'), h_pct=('h_pct', 'median'), area_pct=('area_pct', 'median'),
                                        area_p5=('area_pct', lambda x: x.quantile(.05)), area_p95=('area_pct', lambda x: x.quantile(.95)),
                                        w_px=('w_px', 'median')).round(2).to_string())
print('image sizes per set:', pl.groupby('set')[['img_w', 'img_h']].agg(lambda x: x.value_counts().head(3).to_dict()).to_dict())

                     n  w_pct  h_pct  area_pct  area_p5  area_p95    w_px
set        split                                                         
florida_v6 test     42  12.20   8.11      1.00     0.23      9.17   50.75
           train   914  13.70   9.38      1.22     0.10     20.33   57.00
           valid    92  12.02   7.87      0.93     0.15     11.45   50.00
samrat_v5  test     35  15.62   9.50      1.31     0.54      4.96   65.00
           train   979  13.94   8.77      1.18     0.16      4.55   58.00
           valid    70  15.38   9.44      1.23     0.41      4.38   64.00
usa_v1     test     34  16.19  10.37      1.64     0.42     11.75  103.60
           train  1780  14.34  12.14      1.68     0.33     18.11   91.79
           valid    34  12.54  10.05      1.30     0.33     11.58   80.26
image sizes per set: {'img_w': {'florida_v6': {416: 1048}, 'samrat_v5': {416: 1084}, 'usa_v1': {640: 1848}}, 'img_h': {'florida_v6': {416: 1048}, 'samrat_v5': {416: 1084}, 'usa_v1': {640

## Step 5: near-duplicates
**5a. dHash (perceptual hash).** The image is shrunk to a 9x8 grey thumbnail and each pixel is compared with its right neighbour: 64 yes/no answers. Two images are "near-identical" when at most `DHASH_MAX` = 6 of the 64 answers differ (the PLAN.md rule). It survives resizing and light compression. It does **not** survive rotation, cropping, mirroring or heavy noise, and it can link look-alike scenes of different cars (seen in M1b-1). The hash of the **plate crop** ignores the background, so it can catch the same plate in a different crop; but small crops give noisy hashes, so judge it against its false-alarm rate. dHash needs no ORB work and always runs (5f, 5g).

**5b. ORB (local features) discovers what dHash misses.** ORB finds corner-like keypoints and describes each with a 256-bit code. Two images of the same scene share many keypoints even when one is rotated or cropped. For every cross-split pair inside a set (train x valid, train x test, valid x test) the keypoint codes are matched (ratio test 0.75); pairs with enough matches get a geometry check (RANSAC fits a rotation + scale + shift; "inliers" are matches that agree with one such transform). The inlier count is the pair's score. Descriptors are computed once per image and cached on Drive. **Limits:** ORB matches the same *scene*, so two different photos of the same parked car can match (check the pair sheet); it is not mirror-invariant; heavy noise lowers the count.

**ORB runs in three phases, so a long step never starts by surprise:**
1. `RUN_ORB = False` (default): steps 1-4, dHash and contact sheets only. No ORB work at all.
2. `RUN_ORB = True`, `ORB_LONG = False`: computes the descriptors (cached), **times 200 real pairs per set and prints the minutes** for the controls and the all-pairs search, then stops.
3. `RUN_ORB = True`, `ORB_LONG = True`: runs the controls and the all-pairs search. Both write to Drive CSVs as they go and resume after a disconnect. Every loop prints progress with elapsed time.

In [71]:
PAIRS = [('train', 'valid'), ('train', 'test'), ('valid', 'test')]
ORB_CSV, ORB_DONE, NEG_CSV = Path(REPORTS, 'orb_pairs.csv'), Path(REPORTS, 'orb_done.csv'), Path(REPORTS, 'orb_neg.csv')
T_ORB, FA, DESC = {}, {}, {}  # filled only when the ORB phases run; later cells check for them

def split_names(name, s):
    return list(ok[(ok['set'] == name) & (ok['split'] == s)]['name'])

def orb_describe(path):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    kp, des = cv2.ORB_create(nfeatures=ORB_FEATURES).detectAndCompute(g, None)
    return (np.float32([k.pt for k in kp]), des) if des is not None and len(kp) else (np.zeros((0, 2), np.float32), None)

BF = cv2.BFMatcher(cv2.NORM_HAMMING)
def orb_score(a, b) -> int:
    """Inliers of the best rotation+scale+shift between two images' ORB keypoints (0 if too few matches)."""
    (pa, da), (pb, db) = a, b
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return 0
    good = [m[0] for m in BF.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < ORB_RATIO * m[1].distance]
    if len(good) < ORB_MIN_GOOD:
        return 0  # RANSAC is skipped unless enough ratio-test matches survive
    M, inl = cv2.estimateAffinePartial2D(pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]], method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return int(inl.sum()) if M is not None else 0

def ham(a: int, b: int) -> int:
    return (a ^ b).bit_count()


def neg_pairs(name):
    """The negative-control pair list of one set, redrawn with the fixed seed: (rows of the set, [(i, j), ...]).
    Must stay identical to the draw that produced orb_neg.csv (row k of the csv = pair k of this list)."""
    sub = ok[ok['set'] == name].reset_index(drop=True)
    src, spl = sub['source'].tolist(), sub['split'].tolist()
    r2 = random.Random(SEED + 1); pairs = []
    while len(pairs) < N_NEG:
        i, j = r2.sample(range(len(sub)), 2)
        if src[i] != src[j] and spl[i] != spl[j]:
            pairs.append((i, j))
    return sub, pairs

def _null(scores):
    s = np.sort(np.asarray(scores))[::-1]
    return s, s[int(math.ceil(TRIM_FRAC * len(s))):]  # all scores (high to low), and the null = without the top TRIM_FRAC

def rule_T(scores) -> int:
    """Trimmed rule: drop the highest TRIM_FRAC of the random-pair scores (suspected real duplicates), T = highest remaining + 1, never below ORB_FLOOR."""
    return max(ORB_FLOOR, int(_null(scores)[1][0]) + 1)

def fa_at(scores, t: int) -> dict:
    """False-flag rate at threshold t. raw = share of ALL random pairs >= t (includes real duplicates);
    null = share of the null (top TRIM_FRAC dropped) >= t; fa_rate = null, or the rule-of-three bound 3/N if none reach t."""
    s, null = _null(scores)
    n = int((null >= t).sum())
    return dict(raw=float((s >= t).mean()), null=n / len(null), fa_rate=max(n / len(null), 3 / len(null)), n_trim=len(s) - len(null))

### 5c. Phase 2: descriptors and a timing test on real pairs
Only runs when `RUN_ORB = True`. Descriptors are computed once per image and pickled to Drive (`desc_<set>.pkl`, a file this notebook wrote itself; delete it if you change `ORB_FEATURES`). Then 200 random real pairs per set are timed. The printout gives the measured milliseconds per pair and the minutes for: the negative control, and the all-pairs search in three variants (all three split pairs at 500 features; at 300 features, timed on the first 300 descriptors, an approximation; and only train x test + valid x test). It stops there unless `ORB_LONG = True`.

In [72]:
if not RUN_ORB:
    print('RUN_ORB is False: no ORB work done (phases 2 and 3 are skipped). Steps 1-4, dHash and contact sheets are unaffected.')
else:
    print('OpenCV', cv2.__version__)
    t0 = time.time()
    for name in ZIPS:
        pk = Path(REPORTS, f'desc_{name}.pkl')
        if pk.exists():
            with open(pk, 'rb') as f:
                DESC[name] = pickle.load(f)  # our own cache file
            print(f'{name}: descriptors loaded from {pk.name}'); continue
        items = [(s, n) for s in SPLITS for n in split_names(name, s)]
        d = {}
        for i, (s, n) in enumerate(items, 1):
            d[(s, n)] = orb_describe(ROOT[name] / s / 'images' / n)
            progress(f'descriptors {name}', i, len(items), t0)
        with open(pk, 'wb') as f:
            pickle.dump(d, f)
        DESC[name] = d
    print(f'descriptors ready after {time.time() - t0:.0f} s')
    r3 = random.Random(SEED); tot = {'all500': 0.0, 'all300': 0.0, 'test_only': 0.0, 'neg': 0.0}
    for name in ZIPS:
        keys = list(DESC[name]); k = [r3.sample(keys, 2) for _ in range(200)]
        t0 = time.time(); [orb_score(DESC[name][a], DESC[name][b]) for a, b in k]; per = (time.time() - t0) / 200
        small = {key: (v[0][:300], None if v[1] is None else v[1][:300]) for key, v in DESC[name].items()}
        t0 = time.time(); [orb_score(small[a], small[b]) for a, b in k]; per3 = (time.time() - t0) / 200
        n = {(a, b): len(split_names(name, a)) * len(split_names(name, b)) for a, b in PAIRS}
        kp = np.mean([len(v[0]) for v in DESC[name].values()])
        tot['all500'] += sum(n.values()) * per; tot['all300'] += sum(n.values()) * per3; tot['neg'] += N_NEG * per
        tot['test_only'] += (n[('train', 'test')] + n[('valid', 'test')]) * per
        print(f'{name}: mean keypoints/image {kp:.0f}; {per * 1000:.2f} ms/pair at 500 features ({per3 * 1000:.2f} ms at 300); pairs {sum(n.values()):,} -> {sum(n.values()) * per / 60:.1f} min; negatives {N_NEG * per / 60:.1f} min')
    print(f"ESTIMATE (CPU, +-50%): negative control {tot['neg'] / 60:.1f} min | all-pairs search: 3 split pairs @500 = {tot['all500'] / 60:.1f} min, @300 = {tot['all300'] / 60:.1f} min, train x test + valid x test only @500 = {tot['test_only'] / 60:.1f} min")
    if not ORB_LONG:
        print('Stopped on purpose. Read the estimate; to run the controls and the search set ORB_LONG = True (and optionally lower ORB_FEATURES / delete desc_*.pkl) and re-run from here.')

OpenCV 5.0.0
samrat_v5: descriptors loaded from desc_samrat_v5.pkl
florida_v6: descriptors loaded from desc_florida_v6.pkl
usa_v1: descriptors loaded from desc_usa_v1.pkl
descriptors ready after 0 s
samrat_v5: mean keypoints/image 496; 4.11 ms/pair at 500 features (2.45 ms at 300); pairs 105,245 -> 7.2 min; negatives 1.4 min
florida_v6: mean keypoints/image 494; 7.16 ms/pair at 500 features (2.61 ms at 300); pairs 138,368 -> 16.5 min; negatives 2.4 min
usa_v1: mean keypoints/image 499; 7.68 ms/pair at 500 features (2.92 ms at 300); pairs 123,964 -> 15.9 min; negatives 2.6 min
ESTIMATE (CPU, +-50%): negative control 6.3 min | all-pairs search: 3 split pairs @500 = 39.6 min, @300 = 16.3 min, train x test + valid x test only @500 = 16.4 min


### 5d. Phase 3a: controls (only when `RUN_ORB` and `ORB_LONG`)
Controls come before any threshold is trusted. **Positive:** valid images rotated 5-20 degrees, cropped to 75-90%, with 3% noise, compared with their own original: they should match (and dHash should fail). **Negative:** 20,000 random pairs per set, each from two different splits and with different sources: they should not match. The ORB decision threshold `T_ORB` follows a **trimmed rule**: the highest 0.5% of the negative scores (`TRIM_FRAC`, the top 100 of 20,000) are assumed to be real duplicates and ignored; `T_ORB` is just above the highest score that remains (never below `ORB_FLOOR`). The rule uses ranks, not score sizes, so a pair at 389 inliers counts the same as one at 129 and a handful of true duplicates cannot drag the threshold up. Limit: if more than 0.5% of random pairs are real duplicates the threshold would rise again; the tail tables below make that visible.

**Why this much care:** a set has 100,000+ cross-split pairs, so even a 1-in-1,000 false-alarm rate gives 100+ false flags. From the negatives we estimate a *false-flag rate*: the share of the **null** (the negatives without the suspected duplicates) at or above `T_ORB`; if none reach it, the rate is only known to be below 3/N, the "rule of three" 95% bound, and that bound is used. The raw share of *all* negatives at or above `T_ORB` is printed too (`fa_rate_raw`); it includes real duplicates, so it is an upper bound on false alarms. Both are printed with 5 decimals because the rates are tiny. Step 5f multiplies this rate by the number of pairs to print the **expected false flags** next to the **observed** flags. Why not a percentile or the maximum as the threshold: real duplicates among the random pairs (that is what we hunt) pulled a 99.95th-percentile threshold up to 83 for usa_v1, which rejects 20% of the altered copies in the positive control. Limit: one rate per set, applied to all three split pairs; rates below about 1 in 20,000 cannot be measured.

Each negative pair's score is appended to `orb_neg.csv` on Drive as it is computed. The pair list is drawn with a fixed seed, so after a disconnect the run reads the file and continues from the first pair not yet scored.

In [73]:
if RUN_ORB and ORB_LONG:
    ctrl = []; NEG = {}; POS = {}; t0 = time.time()
    old = pd.read_csv(NEG_CSV, on_bad_lines='skip') if NEG_CSV.exists() else pd.DataFrame(columns=['set', 'orb', 'dh'])
    fresh_neg = not NEG_CSV.exists()
    with open(NEG_CSV, 'a', newline='') as f:
        wr = csv.writer(f)
        if fresh_neg:
            wr.writerow(['set', 'orb', 'dh'])
        for name in ZIPS:
            rng = random.Random(SEED); pos = []
            for s, p in ctrl_images(name):
                im = Image.open(p).convert('RGB')
                v = altered(im, rng, rot=rng.choice([-1, 1]) * rng.uniform(5, 20), crop=rng.uniform(0.75, 0.9), noise=0.03)
                tmp = Path(LOCAL, '_v.png'); v.save(tmp)
                pos.append((orb_score(DESC[name][(s, p.name)], orb_describe(tmp)), ham(dhash(im.convert('L')), dhash(v.convert('L')))))
            print(f'{name}: positive control done ({len(pos)} images), {time.time() - t0:.0f} s elapsed', flush=True)
            sub, pairs = neg_pairs(name)  # fixed seed -> the same list on every run, which makes resuming possible
            spl, nm, dh = sub['split'].tolist(), sub['name'].tolist(), [int(x, 16) for x in sub['dhash']]
            got = old[old['set'] == name][['orb', 'dh']].astype(int).values.tolist()[:N_NEG]
            print(f'{name}: negatives {len(got)}/{N_NEG} already on Drive', flush=True)
            for k in range(len(got), N_NEG):
                i, j = pairs[k]
                s_, d_ = orb_score(DESC[name][(spl[i], nm[i])], DESC[name][(spl[j], nm[j])]), ham(dh[i], dh[j])
                wr.writerow([name, s_, d_]); got.append((s_, d_))
                if (k + 1) % 500 == 0:
                    f.flush()
                progress(f'negatives {name}', k + 1, N_NEG, t0)
            f.flush()
            pos, neg = np.array(pos), np.array(got)
            NEG[name], POS[name] = neg, pos[:, 0]  # kept for the tail tables, the pair view and the override
            t = rule_T(neg[:, 0]); r = fa_at(neg[:, 0], t)
            ctrl.append(dict(set=name, T_ORB=t, pos_n=len(pos), pos_orb_min=pos[:, 0].min(), pos_orb_p5=float(np.percentile(pos[:, 0], 5)), pos_orb_median=float(np.median(pos[:, 0])),
                             pos_orb_recall=float((pos[:, 0] >= t).mean()), pos_dhash_recall=float((pos[:, 1] <= DHASH_MAX).mean()),
                             neg_n=len(neg), n_trimmed=r['n_trim'], fa_rate=r['fa_rate'], fa_rate_raw=r['raw'], neg_ge_T=int((neg[:, 0] >= t).sum()),
                             neg_orb_p99=float(np.percentile(neg[:, 0], 99)), neg_orb_max=neg[:, 0].max(), neg_orb_top5=sorted(neg[:, 0].tolist())[-5:], neg_dhash_le6=float((neg[:, 1] <= DHASH_MAX).mean())))
    CTRL = pd.DataFrame(ctrl).set_index('set')
    print(CTRL.to_string(float_format=lambda x: f'{x:.5f}'))
    T_ORB.update(CTRL['T_ORB'].to_dict())
    FA.update(CTRL['fa_rate'].to_dict())  # false-flag rate per random cross-split pair: null-only share >= T_ORB, or the 3/N bound if none
    print('Read: T_ORB = trimmed rule (top TRIM_FRAC of negatives ignored). fa_rate = false-flag rate used in 5f (null only; 3/N bound if none reach T_ORB); fa_rate_raw = share of ALL negatives >= T_ORB, real duplicates included; n_trimmed = negatives ignored; pos_orb_recall / pos_orb_p5 = share / 5th percentile of the altered-copy scores (T_ORB should sit well below pos_orb_p5); pos_dhash_recall = same for dHash (expect low: this is why ORB is used); neg_orb_p99 / neg_orb_max = 99th percentile / highest random-pair score; neg_ge_T = random pairs at or above T_ORB (suspected duplicates: see 5d-2 and 6b); neg_dhash_le6 = false-alarm rate of dHash.')
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

samrat_v5: positive control done (30 images), 3 s elapsed
samrat_v5: negatives 20000/20000 already on Drive
florida_v6: positive control done (30 images), 7 s elapsed
florida_v6: negatives 20000/20000 already on Drive
usa_v1: positive control done (30 images), 14 s elapsed
usa_v1: negatives 20000/20000 already on Drive
            T_ORB  pos_n  pos_orb_min  pos_orb_p5  pos_orb_median  pos_orb_recall  pos_dhash_recall  neg_n  n_trimmed  fa_rate  fa_rate_raw  neg_ge_T  neg_orb_p99  neg_orb_max               neg_orb_top5  neg_dhash_le6
set                                                                                                                                                                                                                      
samrat_v5      12     30           30    47.70000       124.50000         1.00000           0.00000  20000        100  0.00015      0.00035         7      5.00000           39       [14, 14, 15, 16, 39]        0.00000
florida_v6     12     30 

### 5d-2. Tail of the random-pair scores (the 20,000 negatives)
Per set: how many random pairs score at or above 8, 12, 15, 20, 30, 50 and 80 inliers, and the 20 highest scores. Read it like this: noise ends where the counts stop falling smoothly; a long thin tail of scores far above that is real duplicates. "Draws" counts every draw (a pair can be drawn twice, because the sampler draws with replacement); this is the table used for rates. "Unique pairs" counts each pair once, which is what you look at. The positive-control scores (altered copies of valid images) are listed so you can see where real copies start relative to the noise. The cell first redraws the pair list with the fixed seed and checks that the dHash distance stored for every row equals the one recomputed for that pair; if not, the list does not match `orb_neg.csv` and it stops.

In [74]:
if RUN_ORB and ORB_LONG:
    CUTS = (8, 12, 15, 20, 30, 50, 80)
    raw_rows, uni_rows = [], []
    for name in ZIPS:
        sub, pairs = neg_pairs(name)
        s = NEG[name]; dh = [int(x, 16) for x in sub['dhash']]
        assert len(s) == len(pairs) and all(ham(dh[i], dh[j]) == d for (i, j), d in zip(pairs, s[:, 1])), f'{name}: regenerated pair list does not match orb_neg.csv (SEED, N_NEG or the data changed)'
        first = {}
        for (i, j), sc in zip(pairs, s[:, 0]):
            first.setdefault((min(i, j), max(i, j)), int(sc))
        u = np.array(list(first.values()))
        raw_rows.append(dict(set=name, draws=len(s), **{f'>={c}': int((s[:, 0] >= c).sum()) for c in CUTS}))
        uni_rows.append(dict(set=name, unique_pairs=len(u), **{f'>={c}': int((u >= c).sum()) for c in CUTS}))
        print(f'{name}: top 20 unique-pair scores {sorted(u.tolist(), reverse=True)[:20]}')
        print(f'{name}: positive-control scores, sorted {sorted(POS[name].tolist())}')
    print('draws at or above each cutoff (use for rates):'); print(pd.DataFrame(raw_rows).to_string(index=False))
    print('unique pairs at or above each cutoff:'); print(pd.DataFrame(uni_rows).to_string(index=False))
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

samrat_v5: top 20 unique-pair scores [39, 16, 14, 14, 13, 12, 11, 11, 11, 11, 11, 10, 10, 10, 10, 10, 10, 9, 9, 9]
samrat_v5: positive-control scores, sorted [30, 36, 62, 69, 71, 74, 84, 90, 93, 95, 98, 112, 117, 117, 122, 127, 128, 135, 135, 137, 137, 139, 143, 143, 149, 152, 154, 169, 169, 187]
florida_v6: top 20 unique-pair scores [19, 17, 15, 14, 14, 13, 13, 12, 12, 11, 10, 10, 9, 9, 9, 9, 9, 9, 9, 9]
florida_v6: positive-control scores, sorted [19, 21, 34, 46, 48, 54, 55, 55, 58, 59, 60, 63, 71, 76, 77, 78, 79, 84, 84, 87, 93, 93, 98, 100, 100, 112, 125, 132, 150, 175]
usa_v1: top 20 unique-pair scores [389, 299, 160, 150, 129, 126, 123, 115, 112, 89, 82, 81, 76, 65, 63, 53, 52, 51, 49, 47]
usa_v1: positive-control scores, sorted [34, 45, 53, 66, 67, 73, 85, 90, 93, 94, 95, 104, 112, 115, 117, 120, 121, 122, 122, 122, 128, 132, 134, 141, 150, 151, 154, 162, 170, 199]
draws at or above each cutoff (use for rates):
       set  draws  >=8  >=12  >=15  >=20  >=30  >=50  >=80
 samrat_v

### 5e. Phase 3b: ORB over all cross-split pairs (only when `RUN_ORB` and `ORB_LONG`)
Each query image's pairs are appended to `orb_pairs.csv` (pairs with at least `ORB_KEEP` inliers) and the image is logged in `orb_done.csv`; a re-run skips logged images. Progress prints at least every 30 s.

In [87]:
if RUN_ORB and ORB_LONG:
    done_q = set()
    if ORB_DONE.exists():
        d = pd.read_csv(ORB_DONE, dtype=str, on_bad_lines='skip'); done_q = set(zip(d['set'], d['a'], d['b'], d['name_a']))
    fresh = not ORB_CSV.exists()
    t0 = time.time()
    with open(ORB_CSV, 'a', newline='') as fp, open(ORB_DONE, 'a', newline='') as fd:
        wp, wd = csv.writer(fp), csv.writer(fd)
        if fresh:
            wp.writerow(['set', 'a', 'b', 'name_a', 'name_b', 'inliers']); wd.writerow(['set', 'a', 'b', 'name_a'])
        for name in ZIPS:
            for a, b in PAIRS:
                nb, na_all = split_names(name, b), split_names(name, a)
                for q, na in enumerate(na_all, 1):
                    if (name, a, b, na) not in done_q:
                        for nb_ in nb:
                            s = orb_score(DESC[name][(a, na)], DESC[name][(b, nb_)])
                            if s >= ORB_KEEP:
                                wp.writerow([name, a, b, na, nb_, s])
                        wd.writerow([name, a, b, na])
                        if q % 20 == 0:
                            fp.flush(); fd.flush()
                    progress(f'orb {name} {a}x{b}', q, len(na_all), t0)
                fp.flush(); fd.flush()
                print(f'{name} {a} x {b} done, {time.time() - t0:.0f} s elapsed', flush=True)
    print('ORB search finished')
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

  orb samrat_v5 trainxvalid: 979/979, 0 s elapsed
samrat_v5 train x valid done, 0 s elapsed
  orb samrat_v5 trainxtest: 979/979, 0 s elapsed
samrat_v5 train x test done, 0 s elapsed
  orb samrat_v5 validxtest: 70/70, 0 s elapsed
samrat_v5 valid x test done, 0 s elapsed
  orb florida_v6 trainxvalid: 972/972, 0 s elapsed
florida_v6 train x valid done, 0 s elapsed
  orb florida_v6 trainxtest: 972/972, 0 s elapsed
florida_v6 train x test done, 0 s elapsed
  orb florida_v6 validxtest: 92/92, 0 s elapsed
florida_v6 valid x test done, 0 s elapsed
  orb usa_v1 trainxvalid: 1806/1806, 0 s elapsed
usa_v1 train x valid done, 0 s elapsed
  orb usa_v1 trainxtest: 1806/1806, 0 s elapsed
usa_v1 train x test done, 0 s elapsed
  orb usa_v1 validxtest: 34/34, 0 s elapsed
usa_v1 valid x test done, 0 s elapsed
ORB search finished


### 5e-2. Tail of the FULL search, and a cross-check against the sample
The search in 5e scored **every** cross-split pair, so this is the exact count, not an estimate. Table 1: unique pairs at or above each cutoff, per set and split pair, for pairs whose file-name sources differ (these are the hidden copies) and, separately, for pairs with the same source (the expected augmented copies). The 20,000 random pairs of 5d draw only from the different-source pairs, so only that column is comparable.

Table 2 is the cross-check, per set. The sample says "a share p of random different-source pairs reach cutoff c", so the full search should find about `p x (number of different-source cross-split pairs)` of them: that is the `sample_estimate`. `full_search` is the real count. They should agree within the sampling error (`+-2se`, from the number of sample hits; with few hits it is large). If they do not, either the search or the sample has a problem. The search must be complete for a set (the table says so) or its counts are too low.

In [76]:
if RUN_ORB and ORB_LONG:
    full = pd.read_csv(ORB_CSV, on_bad_lines='skip').drop_duplicates(subset=['set', 'a', 'b', 'name_a', 'name_b'])
    full['same_source'] = full['name_a'].map(source_of) == full['name_b'].map(source_of)
    dqd = pd.read_csv(ORB_DONE, dtype=str, on_bad_lines='skip').drop_duplicates()
    def universe(name, a, b):  # number of different-source cross-split pairs
        ca = Counter(ok[(ok['set'] == name) & (ok['split'] == a)]['source']); cb = Counter(ok[(ok['set'] == name) & (ok['split'] == b)]['source'])
        return sum(ca.values()) * sum(cb.values()) - sum(v * cb.get(k, 0) for k, v in ca.items())
    rows1, complete = [], {}
    for name in ZIPS:
        for a, b in PAIRS:
            n_done = int(((dqd['set'] == name) & (dqd['a'] == a) & (dqd['b'] == b)).sum()); want = len(split_names(name, a))
            complete[(name, a, b)] = n_done == want
            f_ = full[(full['set'] == name) & (full['a'] == a) & (full['b'] == b)]
            for label, part in (('different source', f_[~f_['same_source']]), ('same source', f_[f_['same_source']])):
                rows1.append(dict(set=name, pair=f'{a} x {b}', kind=label, searched=f'{n_done}/{want}', **{f'>={c}': int((part['inliers'] >= c).sum()) for c in CUTS}))
    print('Table 1: full search, unique pairs at or above each cutoff'); print(pd.DataFrame(rows1).to_string(index=False))
    rows2 = []
    for name in ZIPS:
        M = sum(universe(name, a, b) for a, b in PAIRS); ok_all = all(complete[(name, a, b)] for a, b in PAIRS)
        d = full[(full['set'] == name) & (~full['same_source'])]
        for c in CUTS:
            cnt = int((NEG[name][:, 0] >= c).sum()); E = cnt * M / N_NEG; se = math.sqrt(cnt) * M / N_NEG; O = int((d['inliers'] >= c).sum())
            rows2.append(dict(set=name, cutoff=c, universe_pairs=M, sample_hits=cnt, sample_estimate=round(E, 1), plus_minus_2se=round(2 * max(se, M / N_NEG), 1),
                              full_search=O if ok_all else None, agrees=(abs(O - E) <= 2 * max(se, M / N_NEG)) if ok_all else 'search incomplete'))
    print('Table 2: cross-check (different-source pairs, all three split pairs of a set)'); print(pd.DataFrame(rows2).to_string(index=False))
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

Table 1: full search, unique pairs at or above each cutoff
       set          pair             kind  searched  >=8  >=12  >=15  >=20  >=30  >=50  >=80
 samrat_v5 train x valid different source   979/979  188    33    17     5     2     0     0
 samrat_v5 train x valid      same source   979/979    0     0     0     0     0     0     0
 samrat_v5  train x test different source   979/979   86    17     9     2     0     0     0
 samrat_v5  train x test      same source   979/979    0     0     0     0     0     0     0
 samrat_v5  valid x test different source     70/70    3     0     0     0     0     0     0
 samrat_v5  valid x test      same source     70/70    0     0     0     0     0     0     0
florida_v6 train x valid different source   972/972   97    25     8     4     3     3     1
florida_v6 train x valid      same source   972/972 1006   785   652   511   357   170    51
florida_v6  train x test different source   972/972   48     4     0     0     0     0     0
florida_v6 

### 5e-3. Optional threshold override
The rule in 5d sets `T_ORB`. After reading the tail tables you may set `T_ORB_OVERRIDE` in the parameters cell (for example `{'usa_v1': 15}`) and re-run from here. This cell then replaces `T_ORB` for that set and recomputes the false-flag rate **from the negatives at your threshold**, with the same function the rule uses: the share of the null (top 0.5% ignored) at or above your T, or the 3/N bound if none; the raw share of all negatives is printed as well. It also prints the positive-control recall at that T. Re-run 5f and everything after it, because they use `T_ORB` and `fa_rate`. Search results (`orb_pairs.csv`) keep every pair with at least 8 inliers, so any T of 8 or more needs no new search.

In [88]:
if RUN_ORB and ORB_LONG:
    for name, t in T_ORB_OVERRIDE.items():
        assert name in NEG, f'{name}: not a set name, or 5d has not run'
        r = fa_at(NEG[name][:, 0], t); T_ORB[name], FA[name] = t, r['fa_rate']
        print(f'{name}: T_ORB overridden to {t}. Negatives at or above it: {r["raw"]:.5f} of all draws (real duplicates included), {r["null"]:.5f} of the null after dropping the top {r["n_trim"]}. fa_rate used: {r["fa_rate"]:.5f}. Positive-control recall at {t}: {(POS[name] >= t).mean():.2f}')
    print('T_ORB and fa_rate in force:', {n: (T_ORB[n], round(FA[n], 5)) for n in ZIPS})
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

samrat_v5: T_ORB overridden to 42. Negatives at or above it: 0.00000 of all draws (real duplicates included), 0.00000 of the null after dropping the top 100. fa_rate used: 0.00015. Positive-control recall at 42: 0.93
florida_v6: T_ORB overridden to 25. Negatives at or above it: 0.00000 of all draws (real duplicates included), 0.00000 of the null after dropping the top 100. fa_rate used: 0.00015. Positive-control recall at 25: 0.93
usa_v1: T_ORB overridden to 40. Negatives at or above it: 0.00135 of all draws (real duplicates included), 0.00000 of the null after dropping the top 100. fa_rate used: 0.00015. Positive-control recall at 40: 0.97
T_ORB and fa_rate in force: {'samrat_v5': (42, 0.00015), 'florida_v6': (25, 0.00015), 'usa_v1': (40, 0.00015)}


### 5f. Duplicates across splits, per set
For each set: pairs and **images** with a twin in another split, by method (dHash whole image, dHash plate crop, and ORB at `T_ORB` if the ORB phases have run). "Different source name" = the pair's file-name sources differ, i.e. a duplicate the name grouping in step 2 could not see. "Test/valid images with a twin in train" is the number that matters for leakage. For the ORB rows, `n_pairs_searched` counts only the pairs actually searched (logged in `orb_done.csv`), and `expected_false` = false-flag rate x that number.

In [89]:
def hmat(a, b):  # pairwise Hamming distance between two uint64 arrays, via a byte popcount table
    pop = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
    return pop[(a[:, None] ^ b[None, :]).view(np.uint8).reshape(len(a), len(b), 8)].sum(-1)

def hash_pairs(sub_a, sub_b, col):
    ha, hb = sub_a.dropna(subset=[col]), sub_b.dropna(subset=[col])
    A = np.array([int(x, 16) for x in ha[col]], dtype=np.uint64); B = np.array([int(x, 16) for x in hb[col]], dtype=np.uint64)
    if not len(A) or not len(B):
        return []
    i, j = np.where(hmat(A, B) <= DHASH_MAX)
    return [(ha.iloc[x], hb.iloc[y]) for x, y in zip(i, j)]

orb = pd.read_csv(ORB_CSV, on_bad_lines='skip').drop_duplicates() if ORB_CSV.exists() else pd.DataFrame(columns=['set', 'a', 'b', 'name_a', 'name_b', 'inliers'])
dq = pd.read_csv(ORB_DONE, dtype=str, on_bad_lines='skip').drop_duplicates() if ORB_DONE.exists() else pd.DataFrame(columns=['set', 'a', 'b', 'name_a'])
FLAGGED = {}  # set -> list of (method, split_a, name_a, split_b, name_b)
ORBSCORE = {}  # (set, split_a, name_a, split_b, name_b) -> ORB inliers, for ranking
rows = []
for name in ZIPS:
    FLAGGED[name] = []
    for a, b in PAIRS:
        sa, sb = ok[(ok['set'] == name) & (ok['split'] == a)], ok[(ok['set'] == name) & (ok['split'] == b)]
        if sa.empty or sb.empty:
            continue
        found = {}
        found['dhash_full'] = [(x['name'], y['name'], x['source'], y['source']) for x, y in hash_pairs(sa, sb, 'dhash')]
        found['dhash_plate'] = [(x['name'], y['name'], x['source'], y['source']) for x, y in hash_pairs(sa, sb, 'plate_dhash')]
        searched = len(sa) * len(sb)
        if name in T_ORB:
            o = orb[(orb['set'] == name) & (orb['a'] == a) & (orb['b'] == b) & (orb['inliers'] >= T_ORB[name])]
            found['orb'] = [(r.name_a, r.name_b, source_of(r.name_a), source_of(r.name_b)) for r in o.itertuples()]
            ORBSCORE.update({(name, a, r.name_a, b, r.name_b): int(r.inliers) for r in o.itertuples()})
            orb_searched = int(((dq['set'] == name) & (dq['a'] == a) & (dq['b'] == b)).sum()) * len(sb)
        for m, pr in found.items():
            FLAGGED[name] += [(m, a, x, b, y) for x, y, _, _ in pr]
            n_s = orb_searched if m == 'orb' else searched
            rows.append(dict(set=name, pair=f'{a} x {b}', method=m, n_pairs_searched=n_s,
                             expected_false=round(FA[name] * n_s, 1) if m == 'orb' else None, pairs=len(pr), pairs_diff_source=sum(p[2] != p[3] for p in pr),
                             images_a=len({p[0] for p in pr}), of_a=len(sa), images_b=len({p[1] for p in pr}), of_b=len(sb)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
if T_ORB:
    print('Read: for the orb rows, compare pairs (observed flags) with expected_false (false-flag rate x pairs searched). Observed far above expected = real signal; near expected = mostly chance. Expected is an upper-bound style estimate (see 5d).')
else:
    print('NOTE: no orb rows because the ORB phases have not run (RUN_ORB / ORB_LONG); only dHash is shown.')

       set          pair      method  n_pairs_searched  expected_false  pairs  pairs_diff_source  images_a  of_a  images_b  of_b
 samrat_v5 train x valid  dhash_full             68530             NaN      0                  0         0   979         0    70
 samrat_v5 train x valid dhash_plate             68530             NaN      0                  0         0   979         0    70
 samrat_v5 train x valid         orb             68530            10.3      0                  0         0   979         0    70
 samrat_v5  train x test  dhash_full             34265             NaN      0                  0         0   979         0    35
 samrat_v5  train x test dhash_plate             34265             NaN      0                  0         0   979         0    35
 samrat_v5  train x test         orb             34265             5.2      0                  0         0   979         0    35
 samrat_v5  valid x test  dhash_full              2450             NaN      0                  0 

### 5g. Across the three sets (dHash only, as agreed)
Every image of one set against every image of another, all splits. ORB is not run across sets. The Objects In The Wild author says some photos came from another dataset, so this matters most for the Florida set. Same limits as 5a.

In [79]:
rows = []; CROSS = []
names = list(ZIPS)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        sa, sb = ok[ok['set'] == names[i]], ok[ok['set'] == names[j]]
        for col in ('dhash', 'plate_dhash'):
            pr = hash_pairs(sa, sb, col)
            if col == 'dhash':
                CROSS += [(names[i], x['split'], x['name'], names[j], y['split'], y['name']) for x, y in pr]
            rows.append(dict(set_a=names[i], set_b=names[j], hash=col, pairs=len(pr), images_a=len({x['name'] for x, _ in pr}), images_b=len({y['name'] for _, y in pr}),
                             splits=dict(Counter((x['split'], y['split']) for x, y in pr))))
print(pd.DataFrame(rows).to_string(index=False))

     set_a      set_b        hash  pairs  images_a  images_b splits
 samrat_v5 florida_v6       dhash      0         0         0     {}
 samrat_v5 florida_v6 plate_dhash      0         0         0     {}
 samrat_v5     usa_v1       dhash      0         0         0     {}
 samrat_v5     usa_v1 plate_dhash      0         0         0     {}
florida_v6     usa_v1       dhash      0         0         0     {}
florida_v6     usa_v1 plate_dhash      0         0         0     {}


## Step 6: contact sheets (saved to Drive, not the repo)
Seeded random sample of 20 images per set per split (5 x 4), then, per set and split pair, the 30 highest-scoring ORB pairs side by side (best first, score in the title), followed by up to 10 dHash-only pairs. Titles show the split only: no file names, no plate text. **Please look at these yourself**: they are the only way to judge US vs EU, readability, and whether a flagged pair is a real copy or two photos of one scene. Both cells below carry the tag `remove_output`.

In [ ]:
def sheet(items, title, out, ncols=5):
    n = len(items); nrows = max(1, math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.4 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (p, t) in zip(axes.ravel(), items):
        ax.imshow(Image.open(p).convert('RGB')); ax.set_title(t, fontsize=8)
    fig.suptitle(title); fig.savefig(out, dpi=110, bbox_inches='tight'); plt.show(); plt.close(fig)

rng = random.Random(SEED)
for name in ZIPS:
    for s in SPLITS:
        if (ROOT[name] / s / 'images').is_dir():
            imgs = images_of(name, s)
            pick = rng.sample(imgs, min(N_SHEET, len(imgs)))
            sheet([(p, s) for p in pick], f'{name} / {s}: {len(pick)} random of {len(imgs)}', f'{REPORTS}/{name}_{s}.png')

In [ ]:
for name in ZIPS:
    for a, b in PAIRS:
        orb_fl = sorted(((sc, x, y) for (n_, a_, x, b_, y), sc in ORBSCORE.items() if n_ == name and a_ == a and b_ == b), reverse=True)[:N_PAIR_TOP]  # best score first
        items = []
        for sc, x, y in orb_fl:
            items += [(ROOT[name] / a / 'images' / x, f'ORB {sc}: {a}'), (ROOT[name] / b / 'images' / y, f'ORB {sc}: {b}')]
        extra = [(x, y) for m, a_, x, b_, y in FLAGGED[name] if m != 'orb' and a_ == a and b_ == b and (name, a, x, b, y) not in ORBSCORE][:10]  # dHash-only pairs
        for x, y in extra:
            items += [(ROOT[name] / a / 'images' / x, f'dHash: {a}'), (ROOT[name] / b / 'images' / y, f'dHash: {b}')]
        if items:
            sheet(items, f'{name} {a} x {b}: top {len(orb_fl)} ORB pairs by score, then {len(extra)} dHash-only (adjacent images = one pair)', f'{REPORTS}/{name}_pairs_{a}_{b}.png', ncols=4)
        else:
            print(name, a, 'x', b, ': no flagged pairs to draw')

## Step 6b: the highest-scoring random pairs, side by side
The 10 highest-scoring unique pairs among the 20,000 random pairs of one set (`TOP_SET`, usa_v1 by default), drawn from the redrawn pair list. They are random cross-split pairs with different file-name sources, so a high score means a copy with a different name, or one scene photographed twice. The table beside it helps tell which: `dhash_dist` (small = visually identical), and the ORB fit between the two images: **shift** in pixels (x, y), **rotation** in degrees and **scale**. A same-photo copy or a static-camera video frame fits shift near 0, rotation near 0, scale near 1; the same place from another viewpoint does not. (Shift is only a pixel shift when rotation is about 0 and scale about 1.) The refit uses RANSAC again, so its inlier count can differ by a few from the stored score; the ranking uses the stored score. This cell carries the tag `remove_output`; the image is also saved to Drive. No file names are shown.

In [ ]:
TOP_SET, TOP_N = 'usa_v1', 10
def orb_fit(a, b):
    # (inliers, shift_x, shift_y, rotation_deg, scale) of the best rotation+scale+shift between two images; None if no fit
    (pa, da), (pb, db) = a, b
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return None
    good = [m[0] for m in BF.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < ORB_RATIO * m[1].distance]
    if len(good) < 3:
        return None
    M, inl = cv2.estimateAffinePartial2D(pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]], method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return None if M is None else (int(inl.sum()), float(M[0, 2]), float(M[1, 2]), float(np.degrees(np.arctan2(M[1, 0], M[0, 0]))), float(np.hypot(M[0, 0], M[1, 0])))

if RUN_ORB and ORB_LONG:
    sub, pairs = neg_pairs(TOP_SET); s = NEG[TOP_SET]
    first = {}
    for (i, j), (sc, d) in zip(pairs, s.tolist()):
        first.setdefault((min(i, j), max(i, j)), (sc, d))
    top = sorted(first.items(), key=lambda kv: -kv[1][0])[:TOP_N]
    items, rows = [], []
    for rank, ((i, j), (sc, d)) in enumerate(top, 1):
        a, b = (sub['split'][i], sub['name'][i]), (sub['split'][j], sub['name'][j])
        fit = orb_fit(DESC[TOP_SET][a], DESC[TOP_SET][b]) or (None,) * 5
        items += [(ROOT[TOP_SET] / a[0] / 'images' / a[1], f'#{rank} ORB {sc}: {a[0]}'), (ROOT[TOP_SET] / b[0] / 'images' / b[1], f'#{rank} ORB {sc}: {b[0]}')]
        rows.append(dict(rank=rank, orb=sc, splits=f'{a[0]}/{b[0]}', dhash_dist=d, refit_inliers=fit[0], shift_x=fit[1], shift_y=fit[2], rot_deg=fit[3], scale=fit[4]))
    print(pd.DataFrame(rows).round(2).to_string(index=False))
    sheet(items, f'{TOP_SET}: top {len(top)} random pairs by ORB score (adjacent images = one pair)', f'{REPORTS}/top_negative_pairs_{TOP_SET}.png', ncols=4)
else:
    print('Skipped: needs RUN_ORB = True and ORB_LONG = True.')

## Step 7: summary table per set
**Part 1 (numbers):** per set and split: images, distinct sources, sources also present in another split, share of images flagged by each score and by none ("looks unaltered" = no score fired), and cross-split twins (ORB at `T_ORB`, or dHash whole image) found for that split's images against the other splits. **"Looks unaltered" is weaker than "is unaltered".**

**Part 2 (what each score can detect):** the README's augmentation lines are parsed and each is matched to the score that could see it. Anything with "none" cannot be flagged by these three scores, only (partly) by the duplicate search. Flags that fire although **no** README augmentation maps to them are listed as "unexplained" (for example a noise flag on a set whose README lists only exposure and blur).

In [91]:
rows = []
for name in ZIPS:
    for s in SPLITS:
        sub = ok[(ok['set'] == name) & (ok['split'] == s)]
        if sub.empty:
            continue
        twins = {x for m, a, x, b, y in FLAGGED[name] if m in ('orb', 'dhash_full') and a == s} | {y for m, a, x, b, y in FLAGGED[name] if m in ('orb', 'dhash_full') and b == s}
        rows.append(dict(set=name, split=s, images=len(sub), sources=sub['source'].nunique(),
                         sources_also_elsewhere=len(set(sub['source']) & set(ok[(ok['set'] == name) & (ok['split'] != s)]['source'])),
                         sp_flag=round(sub.f_sp.mean(), 3), border_flag=round(sub.f_border.mean(), 3), blur_flag=round(sub.f_blur.mean(), 3),
                         unaltered_looking=round(1 - sub.f_any.mean(), 3), imgs_with_cross_split_twin=len(twins)))
print(pd.DataFrame(rows).to_string(index=False))

       set split  images  sources  sources_also_elsewhere  sp_flag  border_flag  blur_flag  unaltered_looking  imgs_with_cross_split_twin
 samrat_v5 train     979      245                       0    0.000        0.356      0.000              0.644                           0
 samrat_v5 valid      70       70                       0    0.000        0.200      0.000              0.800                           0
 samrat_v5  test      35       35                       0    0.000        0.143      0.000              0.857                           0
florida_v6 train     972       97                      40    0.832        0.614      0.001              0.061                         414
florida_v6 valid      92       41                      29    0.804        0.533      0.000              0.109                          78
florida_v6  test      46       25                      23    0.891        0.848      0.000              0.000                          43
    usa_v1 train    1806      578 

In [84]:
SCORE_OF = [('rotat|shear|skew|tilt', 'border (only if black wedges show; a small rotation or a crop-to-fit may not)'),
            ('noise|salt|pepper', 'salt-and-pepper'), ('blur', 'blur'),
            ('crop|zoom|flip|mirror|bright|exposure|saturat|hue|gray|grey|contrast|cutout|mosaic', 'none (only the duplicate search can help; ORB is not mirror-invariant)')]
rows = []
for name in ZIPS:
    txt = readme(name); lines = txt.splitlines()
    start = next((i for i, l in enumerate(lines) if 'augmentation' in l.lower()), None)
    augs = []
    if start is not None:
        for l in lines[start + 1:]:
            if l.strip().startswith(('*', '-')):
                augs.append(l.strip().lstrip('*- ').strip())
            elif augs and not l.strip():
                break
    if not augs:
        print(f'{name}: NO augmentation lines parsed from the README; read the README printed in step 1 and fill this by hand.')
    covered = set()
    for a in augs:
        m = next((s for pat, s in SCORE_OF if re.search(pat, a, re.I)), 'none (not recognised)')
        covered.add(m.split(' ')[0]); rows.append(dict(set=name, readme_augmentation=a, detected_by=m))
    for col, label in (('f_sp', 'salt-and-pepper'), ('f_border', 'border'), ('f_blur', 'blur')):
        n_flag = int(ok[(ok['set'] == name)][col].sum())
        if label.split(' ')[0] not in covered and n_flag:
            rows.append(dict(set=name, readme_augmentation=f'(none listed for {label})', detected_by=f'UNEXPLAINED: {n_flag} images flagged by the {label} score (natural content or an unlisted change)'))
print(pd.DataFrame(rows).to_string(index=False))

       set                                                           readme_augmentation                                                                                 detected_by
 samrat_v5                     Random exposure adjustment of between -25 and +25 percent                      none (only the duplicate search can help; ORB is not mirror-invariant)
 samrat_v5                                Random Gaussian blur of between 0 and 2 pixels                                                                                        blur
 samrat_v5                                                      (none listed for border) UNEXPLAINED: 368 images flagged by the border score (natural content or an unlisted change)
florida_v6                           Randomly crop between 0 and 20 percent of the image                      none (only the duplicate search can help; ORB is not mirror-invariant)
florida_v6                                Random rotation of between -15 and +15 degrees       

## Before committing this notebook
Three cells show Roboflow images (CC BY, repo is public): the two contact-sheet cells in step 6 and the top-random-pairs cell in step 6b. A fourth cell (step 1, the one starting `import yaml`) is tagged too because its output is only the README text, which can be re-run any time. These four outputs must not be committed. Other text and table outputs stay.
1. Restart & Run All, then save. Check Drive `reports/roboflow_audit/` has the PNGs (including `top_negative_pairs_<set>.png`), `scores.csv`, `orb_pairs.csv`, `orb_neg.csv`.
2. `grep -c '"remove_output"' notebooks/03_roboflow_audit.ipynb` must print 4.
3. From the repo root: `uv run --with nbconvert jupyter nbconvert --to notebook --inplace --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_all_outputs_tags='["remove_output"]' notebooks/03_roboflow_audit.ipynb`
4. `grep -c '"image/png"' notebooks/03_roboflow_audit.ipynb` must print 0.
5. Read `git diff notebooks/03_roboflow_audit.ipynb` for file names, plate text or personal paths before committing.